# PrecepTron × physician ratings — free Kaggle GPU experiment

**Experimental 4-bit quantized transfer test, not an exact reproduction or proof of physician equivalence.** Loads the *real* gated `tbuckley/PrecepTron-32B-Management` LoRA onto the independent `unsloth/Qwen3-32B-bnb-4bit` base. Scores **50 blinded model responses** using the original binary appropriate-uncertainty rubric. Physician labels are never sent to the judge. The comparison step reads them only after all 50 predictions are frozen; the preparation script downloads them separately.

**Before Run All:** use Kaggle **T4 × 2 GPU**, turn **Internet ON**, accept [adapter access](https://huggingface.co/tbuckley/PrecepTron-32B-Management), create a read-only Hugging Face token and store it in Kaggle **Secrets** as `HF_TOKEN`. No token is printed. Quantized checkpoint needs ~20 GB disk, and free GPU availability is not guaranteed.

See [setup guide](https://github.com/KAVentures/ClinicalJudge/blob/main/docs/KAGGLE_FREE_GPU.md).

In [ ]:
# Check free hardware before downloading anything.
from pathlib import Path
import os, shutil, subprocess, sys
try:
    import torch
except ImportError as e:
    raise RuntimeError('Kaggle GPU notebook required (PyTorch missing)') from e
if torch.cuda.device_count() < 2:
    raise RuntimeError('Choose Kaggle Accelerator = GPU T4 x2 before running this notebook.')
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {props.name}, {props.total_memory/2**30:.1f} GiB')
candidates = [p for p in ('/kaggle/temp', '/kaggle/working', '/tmp') if Path(p).is_dir()]
best = max(candidates, key=lambda p: shutil.disk_usage(p).free)
available = shutil.disk_usage(best).free / (2**30)
print(f'Best download volume: {best}, {available:.1f} GiB available')
if available < 21:
    raise RuntimeError('Not enough free disk for the ~19.2 GB quantized checkpoint and overhead. This free Kaggle runtime cannot run the adapter as configured. Do not silently switch models.')
# Persistent only during the Kaggle session; no model checkpoint stored in GitHub.
os.environ['HF_HOME'] = str(Path(best) / 'clinicaljudge_hf_cache')
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
print('Preflight passed')

In [ ]:
# Dependencies for direct quantized PEFT inference. No paid APIs.
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'transformers>=4.53,<5', 'peft>=0.17,<1', 'accelerate>=1.9,<2',
    'bitsandbytes>=0.46,<1', 'huggingface_hub>=0.34,<2',
    'safetensors>=0.5', 'openai>=1.30'], check=True)
print('Inference packages installed')

In [ ]:
# Clone open-source ClinicalJudge and upstream PrecepTron (no model weights in Git).
from pathlib import Path
import subprocess, sys
root = Path('/kaggle/working')
cj = root / 'ClinicalJudge'
pt = root / 'PrecepTron'
for directory, url in [
    (cj, 'https://github.com/KAVentures/ClinicalJudge.git'),
    (pt, 'https://github.com/2v/PrecepTron.git'),
]:
    if not directory.is_dir():
        subprocess.run(['git', 'clone', '--depth', '1', url, str(directory)], check=True)
    else:
        print('Reusing existing checkout', directory)
    revision = subprocess.check_output(['git','-C',str(directory),'rev-parse','HEAD'], text=True).strip()
    print(directory.name, 'git commit:', revision)
subprocess.run([sys.executable, '-m', 'pip', 'install','-q','--no-deps','-e',str(cj)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install','-q','--no-deps','-e',str(pt)], check=True)
sys.path.insert(0, str(cj))
sys.path.insert(0, str(pt))
from preceptron.scorer import build_messages, parse_judge_response
from clinicaljudge.validation import validate
print('ClinicalJudge + PrecepTron scoring code imports OK')

In [ ]:
# Access check: gated adapter must be available to YOUR Hugging Face token.
from huggingface_hub import hf_hub_download
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN').strip()
except Exception:
    pass
if not os.environ.get('HF_TOKEN'):
    raise RuntimeError('Missing Kaggle Secret HF_TOKEN. Add it under Add-ons > Secrets and grant notebook access.')
try:
    path = hf_hub_download('tbuckley/PrecepTron-32B-Management', 'adapter_config.json', token=os.environ['HF_TOKEN'])
except Exception as e:
    raise RuntimeError('Adapter access denied. Accept/request the gate here: https://huggingface.co/tbuckley/PrecepTron-32B-Management') from e
print('Adapter access confirmed. Token is not displayed.')

In [ ]:
# Download only the 50 BLINDED cases and create a SEPARATE human holdout ratings CSV.
# Source commit is frozen inside prepare_healthbench_physician50.py.
from pathlib import Path
import subprocess, sys
DATA_DIR = Path('/kaggle/working/clinicaljudge-healthbench-data')
OUT_DIR = Path('/kaggle/working/clinicaljudge-results')
OUT_DIR.mkdir(exist_ok=True)
subprocess.run([sys.executable,str(cj/'scripts'/'prepare_healthbench_physician50.py'),
                '--output-dir',str(DATA_DIR)], cwd=cj, check=True)
subprocess.run([sys.executable, '-m','clinicaljudge','lint',
                '--input',str(DATA_DIR/'cases_blinded.jsonl')], cwd=cj, check=True)
print('Never open physicians_HOLDOUT.csv before independent model inference finishes.')

In [ ]:
# Pilot on first 5 responses; no physician data are used in scoring.
import json, subprocess, sys
cmd = [sys.executable, str(cj/'scripts'/'run_preceptron_kaggle.py'),
       '--input', str(DATA_DIR/'cases_blinded.jsonl'), '--output-dir', str(OUT_DIR)]
subprocess.run(cmd + ['--limit','5'], cwd=cj, check=True)
preds_file = OUT_DIR/'preceptron_management_quant4_predictions.jsonl'
pilot=[json.loads(l) for l in preds_file.read_text().splitlines() if l.strip()]
valid=sum(p['score'] in (0,1) for p in pilot)
print(f'Pilot: {valid}/5 valid scores')
if valid < 5:
    raise RuntimeError('Pilot had invalid outputs. Inspect results before running all 50; do not impute scores or change model identity.')

In [ ]:
# Run all 50. Resumes the saved five-pilot predictions with the same model/rubric.
subprocess.run(cmd + ['--limit','50'], cwd=cj, check=True)
status=json.loads((OUT_DIR/'run_status.json').read_text())
print(json.dumps(status, indent=2))
if status['scored'] != 50 or status['remaining'] != 0 or status['invalid'] != 0:
    raise RuntimeError('Not all 50 judge predictions succeeded. Results saved for resumption; physician comparison is intentionally unavailable.')

In [ ]:
# Only NOW compare against blinded physicians. Save full audit and reproducibility report.
subprocess.run([sys.executable, str(cj/'scripts'/'compare_preceptron_physician50.py'),
                '--predictions',str(preds_file),
                '--physicians',str(DATA_DIR/'physicians_HOLDOUT.csv'),
                '--output-dir',str(OUT_DIR)], cwd=cj, check=True)
print('Provenance:', OUT_DIR/'model_manifest.json')
print('ClinicalJudge validation:', OUT_DIR/'physician_comparison.json')
print('Per-rater analysis:', OUT_DIR/'preceptron_vs_physicians_summary.md')

In [ ]:
# Zip just the inference outputs and reports (never the HF_TOKEN or original study data).
import zipfile
archive=Path('/kaggle/working/clinicaljudge-physician50-results.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for f in OUT_DIR.rglob('*'):
        if f.is_file():
            z.write(f, arcname=f.name)
print('Saved:', archive)
try:
    from IPython.display import FileLink, display
    display(FileLink(str(archive)))
except ImportError:
    pass